# Week 3, Lab 5 — Mini-project: research, draft, review


In [1]:
import zipfile
import os

zip_path = "/content/shared.zip"      # Path of the uploaded ZIP file
extract_path = "/content/shared"      # Folder where files will be extracted

# Create the folder if it doesn't exist
os.makedirs(extract_path, exist_ok=True)

# Unzip
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ ZIP extracted successfully!")
print("Files extracted to:", extract_path)


✅ ZIP extracted successfully!
Files extracted to: /content/shared


In [2]:
import zipfile
import os

zip_path = "/content/shared.zip"
extract_path = "/content"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ Extracted successfully!")

✅ Extracted successfully!


In [3]:
WEEK = 'Week 3'
LAB = 'Lab 5 — mini-project'

import sys
from pathlib import Path

def _course_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "shared" / "course_runtime.py").exists():
            return p
    for c in [
        here / "agentic_ai_local",
        Path("/content/agentic_ai_local"),
        Path("/content"),
    ]:
        if (c / "shared" / "course_runtime.py").exists():
            return c
    return here

ROOT = _course_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from shared.course_runtime import (
    detect_backend,
    print_banner,
    local_chat,
    calculator,
    lookup_fact,
    today_date,
    extract_json_object,
    parse_tool_call,
    openai_client_kwargs,
    get_langchain_llm,
    TOOL_SCHEMAS,
    MOCK_KB,
)

BACKEND = print_banner(WEEK, LAB)
print("If import failed, unzip/clone the WHOLE course folder (not a single notebook).")


Week 3 / Lab 5 — mini-project
Environment: Google Colab
Backend: huggingface
Tip: Runtime → Change runtime type → T4 GPU for faster generation.
If import failed, unzip/clone the WHOLE course folder (not a single notebook).


In [4]:
if BACKEND == "huggingface":
    %pip install -q transformers torch accelerate fastapi uvicorn crewai
else:
    %pip install -q crewai ollama


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 5.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 198.9/198.9 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.5/42.5 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.9/19.9 MB 64.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.4/269.4 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.0/48.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9

In [ ]:
from crewai import LLM

GEMINI_API_KEY = "your Gemini API key here"  # Replace with your actual Gemini API key

llm = LLM(
    model="gemini/gemini-3.6-flash",
    api_key=GEMINI_API_KEY
)

print("CrewAI is using Gemini 3.6 Flash")

CrewAI is using Gemini 3.6 Flash


In [7]:
from crewai import Agent, Task, Crew, Process
from crewai.tools import BaseTool

# ------------------ Local Knowledge Base ------------------

def lookup_fact(topic: str) -> str:
    kb = {
        "mcp": (
            "MCP (Model Context Protocol) is an open protocol that lets AI models connect "
            "to external tools, APIs, databases, and applications through a standardized interface."
        ),
        "crewai": (
            "CrewAI is a multi-agent framework where agents collaborate using roles, goals, "
            "tasks, tools, and workflows."
        ),
    }

    topic = topic.lower()
    if "mcp" in topic and "crewai" in topic:
        return f"- {kb['mcp']}\n- {kb['crewai']}"
    return kb.get(topic, "No information found.")

# ------------------ Custom Tool ------------------

class LookupTool(BaseTool):
    name: str = "lookup_fact"
    description: str = "Local KB lookup."

    def _run(self, topic: str) -> str:
        return lookup_fact(topic)

# ------------------ Agents ------------------

researcher = Agent(
    role="Researcher",
    goal="Gather facts with the lookup tool.",
    backstory="Analyst.",
    llm=llm,
    tools=[LookupTool()],
)

writer = Agent(
    role="Writer",
    goal="Draft a 120-word student explainer.",
    backstory="Teacher.",
    llm=llm,
)

reviewer = Agent(
    role="Reviewer",
    goal="Check factuality against the research notes and return a corrected final draft.",
    backstory="Strict editor.",
    llm=llm,
)

# ------------------ Tasks ------------------

topic = "MCP"

t1 = Task(
    description=f"Look up facts about {topic} and CrewAI using the lookup_fact tool.",
    expected_output="Bullet points from the tool.",
    agent=researcher,
)

t2 = Task(
    description="Draft a beginner-friendly explanation in about 120 words using the research.",
    expected_output="One short essay.",
    agent=writer,
)

t3 = Task(
    description="Review the draft, correct any factual mistakes, and return ONLY the final student-facing version.",
    expected_output="Final corrected draft.",
    agent=reviewer,
)

# ------------------ Crew ------------------

crew = Crew(
    agents=[researcher, writer, reviewer],
    tasks=[t1, t2, t3],
    process=Process.sequential,
)

# ✅ Google Colab / Jupyter
result = await crew.kickoff_async()
print(result)

ERROR:root:Google Gemini API error: 503 - This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.
ERROR:crewai.flow.runtime:Error executing listener call_llm_and_parse: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}


Imagine building a digital dream team. CrewAI is a framework that helps you do just that. It allows multiple AI agents to collaborate like a real-world workforce, assigning each agent specific roles, goals, tasks, and tools to complete complex workflows together. 

Now, how do these agents interact with the outside world? That is where the Model Context Protocol, or MCP, comes in. MCP is an open protocol that acts like a universal adapter. It allows AI models to connect seamlessly to external tools, APIs, databases, and applications through a standardized interface. 

In short, CrewAI organizes how your AI agents work together, while MCP standardizes how they connect to and interact with external systems, creating powerful, interconnected AI systems.


## Rubric

Three roles, at least one tool call, a reviewer pass, no paid API key.
